# LangSmith Observability for RAG



In [ ]:
%pip install -U langchain langchain-ollama langchain-chroma langchain-community pypdf --quiet

Import Langsmith variables

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()
# LANGCHAIN_API_KEY=
# LANGCHAIN_TRACING_V2=true
# LANGCHAIN_ENDPOINT=https://api.smith.langchain.com
os.environ["LANGCHAIN_PROJECT"] = "langsmith-observe3"

Import libraries

In [8]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

Initializing the models: main LLM and embeddings model

In [9]:
llm = ChatOllama(model="gemma4:e4b", temperature=0)
embeddings = OllamaEmbeddings(model="embeddinggemma")

Ingest files from the data directory

In [3]:
loader = PyPDFDirectoryLoader("./data")
docs = loader.load()
print(f"Loaded {len(docs)} documents") #Langchain counts pages not documents :)

Loaded 5 documents


Split the do documents into smaller chunks. 1000 chars per chunk is a good practice

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

Split into 14 chunks


Initialize ChromaDB with local storage for persistence and create a retriever

In [10]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
#Retrieve 3 chunks
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

Create a prompt to link together the retrieved chunks and the user's prompt

In [11]:
template = """You are a specialized AI assistant. Use the provided documentation 
to answer the user's request. If the answer isn't in the context, be honest.

Context: {context}

Question: {question}

Helpful Answer:"""

custom_prompt = PromptTemplate(
    template=template, 
    input_variables=["context", "question"]
)

## Chain to create the answer only
First step runs two parallel processes to build a dictionary with the two keys the custom prompt is looking for. 

In [12]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | custom_prompt 
    | llm 
    | StrOutputParser()
)

In [13]:
print(type(rag_chain))

<class 'langchain_core.runnables.base.RunnableSequence'>


Test the chain

In [16]:
query = "Explain briefly how solar powerworks"
response = rag_chain.invoke(query)

In [17]:
print(response)

Solar power works by tapping into the nearly limitless energy from the sun, transforming sunlight into usable electricity.

At its core, the process involves capturing the energy contained within **photons** (massless packets of light). The most common method is **photovoltaic (PV) technology**, which relies on the principles of semiconductor physics.

In this process:
1.  Specialized materials, most often **silicon**, are used to create an electrical current.
2.  A solar panel (or module) is constructed from multiple silicon cells arranged in series.
3.  When photons strike these cells, they generate an electrical current.

Overall, solar energy provides a clean, renewable alternative to fossil fuels.


Langsmith did the tracing. The LLM chat was the major contributor because it took long for Ollama to load up

![Langsmith tracing a RAG pipeline](images/langsmith-rag1.png)